# Fixed microenvironment simulation
Both notebooks call the same V1002 generator. All truth labels below are used only for post-training audit.

In [ ]:
import sys
from pathlib import Path
ROOT = Path('/home/xueshuailin/CCC_Phe')
V1002_SOURCE = str(ROOT/'V1002/src')
if V1002_SOURCE in sys.path: sys.path.remove(V1002_SOURCE)
sys.path.insert(0,V1002_SOURCE)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from scipy.optimize import linear_sum_assignment
from phenoniche.v1002.lr_atlas import load_lr_atlas
from phenoniche.v1002.final_simulation import build_final_spec, simulate_final_spatial, simulate_final_bulk, bulk_potential, alr, TRUE_BETA
from phenoniche.v1002.identifiability_run import cox_fit
SEED=40700; PURITY=.5; NICHE_SIZE=100; NOISE=.05
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'


V1002 keeps the flat nonnegative matrix-factorization baseline: C and I share six factors.

## Generate one fixed local-microenvironment dataset

In [ ]:
spec=build_final_spec(load_lr_atlas(str(ROOT/'V1002/data/commuspace_human_lr_atlas.tsv')))
sim=simulate_final_spatial(spec,purity=PURITY,niche_size=NICHE_SIZE,noise=NOISE,seed=SEED)
feature_mask=sim.final_mask.reshape(-1)
C=sim.cs
I=sim.communication[:,feature_mask]
macro_truth=np.array([0,1,1,2,3,4])[sim.labels]
assert C.shape==(1800,8) and sim.cell_types.shape==(1800,48)
assert I.shape[1]==int(feature_mask.sum()) and np.allclose(C.sum(1),1)


## Observable identifiability audit, before any model training

In [ ]:
def pair_auc(x,labels,a,b,core):
    keep=core & np.isin(labels,[a,b]); x=x[keep]; y=(labels[keep]==b).astype(int)
    cv=StratifiedKFold(3,shuffle=True,random_state=SEED)
    scores=np.empty(len(y))
    for tr,te in cv.split(x,y):
        direction=x[tr][y[tr]==1].mean(0)-x[tr][y[tr]==0].mean(0)
        scores[te]=x[te]@direction
    auc=roc_auc_score(y,scores)
    return max(auc,1-auc)
core=sim.rho>=.8
observable=pd.DataFrame([{'Pair':name,'Composition AUC':pair_auc(C,sim.labels,a,b,core),
                          'CCC AUC':pair_auc(I,sim.labels,a,b,core)}
                         for name,a,b in [('N1/N2',1,2),('N3/N4',3,4)]])
display(observable.round(3))


## Raw composition five-state sanity

In [ ]:
raw_km=KMeans(n_clusters=5,n_init=20,random_state=SEED).fit(C)
raw_pred=raw_km.labels_
raw_summary={'ARI':adjusted_rand_score(macro_truth,raw_pred),
             'silhouette':silhouette_score(C,raw_pred)}
fig,axes=plt.subplots(1,2,figsize=(10,4),constrained_layout=True)
for ax,values,title in zip(axes,[macro_truth,raw_pred],['Truth macro states','Raw C KMeans (K=5)']):
    ax.scatter(sim.coordinates[:,0],sim.coordinates[:,1],c=values,s=9,cmap='tab10',vmin=0,vmax=5)
    ax.set_title(title); ax.set_aspect('equal'); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
plt.show()
display(pd.DataFrame([raw_summary]).round(3))


## Outcome-blind CCC filter and views

In [ ]:
display(pd.DataFrame([{'Measurable LR':len(spec.atlas),'Candidate CCC':sim.final_mask.size,
                       'Coverage passed':int(sim.coverage_mask.sum()),'Selected CCC':int(feature_mask.sum()),
                       'C shape':str(C.shape),'I shape':str(I.shape)}]))


In [ ]:
def align_clusters(pred,truth,k=6):
    counts=np.array([[(pred==i).__and__(truth==j).sum() for j in range(k)] for i in range(k)])
    rows,cols=linear_sum_assignment(-counts)
    lookup=dict(zip(rows,cols))
    return np.array([lookup[v] for v in pred]), lookup

def st_metrics(name,pred,truth=sim.labels):
    aligned,_=align_clusters(pred,truth)
    row={'Method':name,'ARI':adjusted_rand_score(truth,pred),'NMI':normalized_mutual_info_score(truth,pred)}
    row.update({label:np.mean(aligned[truth==k]==k) for k,label in enumerate(['BG','N1','N2','N3','N4','N5'])})
    return row,aligned


## C-only, I-only and full NMF

In [ ]:
from phenoniche.v1002.final_model import fit_balanced
from phenoniche.v1002.simulation_model import infer_activity
models={}
for name,blocks in [('C-only',{'HC':C}),('I-only',{'HI':I}),('C+I',{'HC':C,'HI':I})]:
    fit=fit_balanced(blocks,seed=SEED,iterations=400,device=DEVICE)
    w=fit.W.numpy(); w=w/np.maximum(w.sum(1,keepdims=True),1e-12); pred=w.argmax(1)
    metrics,aligned=st_metrics(name,pred)
    models[name]={'fit':fit,'W':w,'pred':pred,'aligned':aligned,'metrics':metrics}


## Spatial comparison and compact metrics

In [ ]:
fig,axes=plt.subplots(1,4,figsize=(16,4),constrained_layout=True)
for ax,(name,values) in zip(axes,[('Truth',sim.labels),('Raw C',raw_pred),
                                  ('I-only NMF',models['I-only']['aligned']),('C+I NMF',models['C+I']['aligned'])]):
    ax.scatter(sim.coordinates[:,0],sim.coordinates[:,1],c=values,s=8,cmap='tab10',vmin=0,vmax=5)
    ax.set_title(name); ax.set_aspect('equal'); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
plt.show()
rows=[item['metrics'] for item in models.values()]
metrics=pd.DataFrame(rows)
comp_macro=np.array([0,1,1,2,3,4])[models['C-only']['aligned']]
metrics['Macro ARI']=np.nan
metrics.loc[metrics.Method=='C-only','Macro ARI']=adjusted_rand_score(macro_truth,comp_macro)
display(metrics.round(3))


## Shared real bulk, phenotype-blind projection and ALR Cox

In [ ]:
pi_true,CB,bulk_expr,time,event,beta,eta=simulate_final_bulk(spec,sim.hc_truth,NOISE,SEED+10,patients=320)
IB=bulk_potential(bulk_expr,spec)[:,feature_mask]
assert IB.shape[1]==I.shape[1]
fit=models['C+I']['fit']; H={name:value.numpy() for name,value in fit.dictionaries.items()}
bulk_blocks={'HC':CB,'HI':IB}
w_bulk=infer_activity(bulk_blocks,{name:H[name] for name in bulk_blocks},steps=200).numpy()
pi_pred=np.maximum(w_bulk,1e-6); pi_pred/=pi_pred.sum(1,keepdims=True)
# Cluster-to-truth permutation is applied only for simulation reporting.
factor_truth=np.array([[np.corrcoef(models['C+I']['W'][:,j],sim.ws_truth[:,k])[0,1] for k in range(6)] for j in range(6)])
factor_rows,truth_cols=linear_sum_assignment(-factor_truth)
order=np.empty(6,dtype=int); order[truth_cols]=factor_rows
pi_pred=pi_pred[:,order]
cox_rows=[]
for source,pi in [('True exposure',pi_true),('Predicted exposure',pi_pred)]:
    result=cox_fit(alr(pi),time,event); gamma=np.array(result['gamma'])
    cox_rows.append({'Source':source,'Risk gamma':gamma[0],'Protective gamma':gamma[2],
                     'Neutral mean |gamma|':np.abs(gamma[[1,3,4]]).mean(),
                     'Validation C':result['validation_C'],'Test C':result['test_C']})
display(pd.DataFrame(cox_rows).round(3))


## Summary
Inspect the observable audit and compact tables. No result files are written; this notebook is intended for manual execution.